# Tarea 10 · Estadística descriptiva de antigüedad y cargos

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/10_estadistica_descriptiva.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Julián.

Resumimos la antigüedad (`tenure`), el cargo mensual (`MonthlyCharges`) y el cargo acumulado (`TotalCharges`). El resultado es una tabla con una interpretación por variable.

## Cómo ejecutarlo

- **Local:** instalá las dependencias de `requirements.txt`, abrí el notebook desde el repo y ejecutá todas las celdas.
- **Colab:** ejecutá todas las celdas. El notebook lee los datos desde GitHub. El botón apunta a `main` y estará disponible cuando se integre la tarea. Para probarla antes, subí el archivo `.ipynb` a Colab.

## 1. Carga del dataset

Usamos `telco_clean.csv` cuando esté disponible. Mientras tanto, usamos `telco_provisorio.csv` y mostramos qué archivo se leyó. Las estadísticas de cada variable se calculan con sus valores disponibles, así que el resumen informa cuántos faltantes se excluyen.

In [ ]:
from pathlib import Path
from urllib.error import HTTPError

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

variables = ["tenure", "MonthlyCharges", "TotalCharges"]
ruta_limpia = Path("data/processed/telco_clean.csv")
ruta_provisoria = Path("data/processed/telco_provisorio.csv")

if EN_COLAB:
    url_repo = "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/"
    fuente = url_repo + ruta_limpia.as_posix()
    try:
        datos = pd.read_csv(fuente, usecols=variables)
    except HTTPError as error:
        if error.code != 404:
            raise
        fuente = url_repo + ruta_provisoria.as_posix()
        datos = pd.read_csv(fuente, usecols=variables)
else:
    actual = Path.cwd().resolve()
    raiz = next(
        (ruta for ruta in (actual, *actual.parents)
         if (ruta / ruta_limpia).is_file() or (ruta / ruta_provisoria).is_file()),
        None,
    )
    if raiz is None:
        raise FileNotFoundError("Falta el CSV de la tarea 3 o el dataset limpio de la tarea 9.")
    fuente = raiz / ruta_limpia
    if not fuente.is_file():
        fuente = raiz / ruta_provisoria
    datos = pd.read_csv(fuente, usecols=variables)

datos = datos[variables]
print(f"Fuente: {fuente}")

## 2. Cálculo de las estadísticas

La media es el promedio, la mediana divide los valores ordenados en dos mitades y la moda es el valor más frecuente. Si hay un empate, mostramos todas las modas. El desvío estándar muestral y el rango describen la dispersión. El rango intercuartílico (IQR = Q3 − Q1) mide cuánto abarca el 50 % central de los datos.

La asimetría indica hacia qué lado se extiende más la distribución. Para la curtosis usamos el exceso de Fisher, que toma el valor 0 como referencia para una distribución normal.

In [ ]:
resumen = datos.agg(["count", "mean", "median", "std", "skew", "kurt"]).T.rename(columns={
    "count": "Datos válidos",
    "mean": "Media",
    "median": "Mediana",
    "std": "Desvío estándar",
    "skew": "Asimetría",
    "kurt": "Exceso de curtosis",
})
resumen["Datos válidos"] = resumen["Datos válidos"].astype(int)
resumen["Faltantes"] = datos.isna().sum()
resumen["Moda"] = [", ".join(map(str, datos[columna].mode())) for columna in variables]
resumen["Rango"] = datos.max() - datos.min()
resumen["IQR"] = datos.quantile(0.75) - datos.quantile(0.25)

resumen = resumen[[
    "Datos válidos", "Faltantes", "Media", "Mediana", "Moda",
    "Desvío estándar", "Rango", "IQR", "Asimetría", "Exceso de curtosis",
]]
resumen.round(2)

## 3. Interpretación de la versión provisoria

- **Antigüedad (`tenure`).** La mediana es de 29 meses y la media de 32,37. La moda es 1 mes, mientras que el 50 % central abarca 46 meses. Esto muestra que conviven clientes recién incorporados y clientes con bastante antigüedad. Conviene distinguir estos grupos al estudiar la permanencia.
- **Cargo mensual (`MonthlyCharges`).** La mediana es de 70,35 y el promedio de 64,76. La asimetría es levemente negativa y la moda es 20,05. El IQR de 54,35 muestra diferencias relevantes entre los cargos que paga el grueso de la cartera.
- **Cargo acumulado (`TotalCharges`).** La media de 2.283,30 supera la mediana de 1.397,48 y la asimetría es positiva (0,96). Los importes altos elevan el promedio, por lo que la mediana ayuda a describir el cargo acumulado de un cliente típico. Este cálculo usó 7.032 casos con datos. Los 11 faltantes del CSV provisorio se excluyeron únicamente de las estadísticas de esta columna.

Revisar estas cifras contra la tabla cuando el notebook lea `telco_clean.csv`.